# 1. Normal and DP

In [ ]:
from torch.amp import autocast
import torch
import torch.nn as nn


layer1 = nn.Linear(3, 3, bias=False, dtype=torch.float32).cuda()
input_data = torch.randn(1, 3).cuda()

print(f'=' * 50)
print(f'Original Weight before the autocast: {layer1.weight.dtype}')
print(f'Dtype of the input before the autocast: {input_data.dtype}')
print(f'=' * 50)

with autocast(device_type='cuda', dtype=torch.bfloat16):
    # Ever operations here gonna take the weights and 'turn' into the dtype that we pass, but don't will change the original value
    output = layer1(input_data)
    print(f'Dtype of the result inside the autocast: {output.dtype}')

print(f'=' * 50)
print(f'Original weight after the autocast: {layer1.weight.dtype}')
print(f'Dtype of the input after the autocast: {input_data.dtype}')


Original Weight before the autocast: torch.float32
Dtype of the input before the autocast: torch.float32
Dtype of the result inside the autocast: torch.bfloat16
Original weight after the autocast: torch.float32
Dtype of the input after the autocast: torch.float32


# 2. FSDP

In [ ]:
from torch.distributed.fsdp.mixed_precision import MixedPrecision
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

# Define the policy of mp
mp_policy = MixedPrecision(
    param_dtype=torch.float32, 
    reduce_dtype=torch.bfloat16, 
    buffer_dtype=torch.bfloat16
)

# Do the model and FSDP
model = None
model = FSDP(model, mixed_precision=mp_policy)

# ...
# All the logic here
# ...

with autocast(device_type='cuda', dtype=torch.bfloat16):
    pass
    # logic of train here

# 3. GPUs and BF16

Some GPUs don't have support for the `bfloat16`, so we need to check first before we use

In [1]:
import torch

print(True if torch.cuda.is_bf16_supported else False)

True
